# Hands-On Lab — Train Your Own Fraud Detection Model on HPE Private Cloud AI

Welcome! In the next ~30 minutes you will do **the same thing the showcase team did** to build the fraud-detection model that is already deployed in HPE MLIS — except this time **you do it yourself**, end-to-end, and the model you train will live in your own MLflow experiment named after you.

## What you will accomplish

By the end of this notebook you will have:

1. **Touched a real-world dataset** — 24 million synthetic credit-card transactions from IBM TabFormer
2. **Seen the RAPIDS speedup** — the same group-by operation, on pandas vs. on cuDF (GPU dataframe)
3. **Submitted a training Job to Kubernetes** — the production MLOps pattern that HPE PCAI is built around
4. **Trained a Graph Neural Network + XGBoost** model for fraud detection — NVIDIA's published blueprint, unchanged
5. **Logged everything to MLflow** with full lineage — params, metrics, artifacts, registered model
6. **Found your trained model in the MLflow Model Registry**, named after you

## What you do **not** have to do

- ❌ Write any model code
- ❌ Author any Kubernetes YAML
- ❌ Set up any infrastructure
- ❌ Configure any auth

You are driving NVIDIA's open-source fraud-detection blueprint on HPE PCAI's managed services. The notebook does all the heavy lifting. **Just run the cells in order, top to bottom.**

## How long this takes

| If you are… | Expected time |
|---|---|
| The first student whose Job lands on a given GPU node | ~25 min (5–10 min one-time spent pulling NVIDIA's 12 GB training image) |
| Anyone whose Job lands on a node where the image is already cached | ~12 min |

Don't worry about which case you're in — both produce the exact same result. The image pull is a one-time thing per node.

## A note for the pre-sales audience

Every cell in this notebook has a short markdown explanation **above it** in plain English. If you are new to ML, MLOps, or Kubernetes, **read the markdown first, then run the code**. The goal is for you to understand *what is happening on the platform*, not just to watch numbers go by.

**The PCAI value-prop is in the connective tissue.** Every step you see — submitting a training Job, mounting shared storage, logging to MLflow, pulling NVIDIA's image with one secret — would be 4 different vendor tools and a month of YAML on bare cloud. Here it's one platform, one login, one notebook.

Let's go.

---

## Section 1 — Your student number

The only thing in this notebook you need to edit is the cell below. Find the **number** assigned to your notebook server (e.g., if your notebook is named `student-3`, your number is `3`) and set it.

**Why this matters:** the notebook uses your student number to namespace everything — your working directory on shared storage, your MLflow experiment, your registered model. So everyone in the room can work in parallel without stepping on each other's runs.


In [ ]:
# ╔══════════════════════════════════════════╗
# ║  EDIT THIS LINE — set your student number ║
# ╚══════════════════════════════════════════╝

STUDENT_ID = 1   # ← change to your number (1 through 8)

# ──────────────────────────────────────────────
# Everything below is automatic. Just keep running cells.

assert isinstance(STUDENT_ID, int) and 1 <= STUDENT_ID <= 99, "STUDENT_ID must be an integer 1-99"
print(f"OK — you are student {STUDENT_ID}. Your MLflow experiment will be: student-{STUDENT_ID}-fraud-detection")

## Section 2 — Install the Python packages we need

We do this BEFORE anything else because some pip installs trigger a kernel restart — and a restart would wipe out any variables we'd defined earlier. So pip install first, set variables after.

This notebook itself needs only a few packages beyond what the PCAI notebook image already ships with:

- **cuDF** (`cudf-cu12`) — RAPIDS' GPU-accelerated pandas-equivalent. We'll use it later for the speed-comparison demo.
- **MLflow** — the experiment tracker. Usually already installed in PCAI notebooks; this is a safety upgrade.
- **boto3** — Python's S3 client. We use it to upload model artifacts to local-s3 (the in-cluster object store backing MLflow).

**Note:** the *training* itself doesn't run in this notebook — it runs inside NVIDIA's container as a Kubernetes Job (Section 10). So we don't need to install PyTorch, cuGraph, torch-geometric, etc. — those are already inside NVIDIA's image.

**If Jupyter prompts you to restart the kernel after this install:** click *Restart*. Then re-run Section 1's `STUDENT_ID` cell (it'll re-read your edit), re-run this Section 2 cell (will be a no-op since packages are already installed), and continue. Kernel restart wipes Python variables but not your cell edits.

This cell takes ~1-2 min.

In [ ]:
%pip install --quiet \
    cudf-cu12==25.4.0 \
    --extra-index-url https://pypi.nvidia.com

%pip install --quiet \
    mlflow>=2.14 \
    boto3>=1.34 \
    pyyaml

print("Dependencies installed.")
print("If Jupyter prompted you to restart the kernel above, restart it now,")
print("then re-run Section 1's STUDENT_ID cell, then re-run this cell (it'll be a no-op).")

## Section 3 — Verify your environment

This cell prints out what your notebook pod sees:

- Your **Kubernetes namespace** (this is your private workspace in the cluster — your training Job will run here, your MLflow runs are logged from here)
- Your **assigned GPU** (1× NVIDIA L40s, visible inside the pod)
- Your **working directory on shared storage** — `/mnt/shared/lab-student-N/`. The shared PVC is mounted into every notebook AND will be mounted into your training Job, so it is the data interchange between them.
- Your **MLflow tracking URL** — PCAI auto-injects this. You don't have to set it.

**Layman tip:** A *pod* is one running instance of your workload on Kubernetes. Each student gets their own pod. A *namespace* is a private workspace — your pod and Jobs live in one namespace; the next student's live in theirs.

In [ ]:
import os, subprocess

# Detect the K8s namespace from the pod's service-account token
with open('/var/run/secrets/kubernetes.io/serviceaccount/namespace') as f:
    NS = f.read().strip()

# Your private working directory on the shared PVC
LAB_DIR = f"/mnt/shared/lab-student-{STUDENT_ID}"
os.makedirs(LAB_DIR, exist_ok=True)
os.makedirs(f"{LAB_DIR}/raw",    exist_ok=True)
os.makedirs(f"{LAB_DIR}/scripts", exist_ok=True)

# MLflow tracking URI is auto-injected by PCAI's notebook image. Falls back if missing.
MLFLOW_URI = os.environ.get('MLFLOW_TRACKING_URI',
                            'http://mlflow.mlflow.svc.cluster.local:5000')
S3_ENDPOINT = 'http://local-s3-service.ezdata-system.svc.cluster.local:30000'

print(f"Namespace:          {NS}")
print(f"Working directory:  {LAB_DIR}")
print(f"MLflow URL:         {MLFLOW_URI}")
print(f"S3 endpoint:        {S3_ENDPOINT}")
print("\n--- GPU info ---")
subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                '--format=csv'], check=False)

## Section 4 — Download the dataset

The dataset is **IBM TabFormer** — a publicly-released synthetic dataset of credit-card transactions, designed by IBM Research as a fraud-detection benchmark. "Synthetic" means it was generated programmatically; no real customer data is involved.

It has **24,386,900 rows** spanning ~28 years (1991–2020), with columns like:

| Column | What it means |
|---|---|
| User, Card | Which cardholder and which of their cards |
| Year, Month, Day, Time | When the transaction happened |
| Amount | How much (in USD) |
| Use Chip | Chip / swipe / online |
| Merchant Name, Merchant City, Merchant State, Zip | Where the transaction happened |
| MCC | Industry code of the merchant (4-digit standard) |
| Errors? | Any errors during the transaction (chip read fail, etc.) |
| **Is Fraud?** | The label we want to predict — Yes or No |

The compressed file (`transactions.tgz`) is ~266 MB. Your instructor has hosted it at a stable URL — we download it now into your private working directory.

**Layman tip:** *Fraud is rare* in this dataset — under 0.2% of raw rows. That's representative of real-world data and is part of what makes fraud detection hard.

In [ ]:
# Hosted on the showcase repo as a GitHub release asset.
# If you need to update the dataset version later, change the tag in the URL.
TRANSACTIONS_TGZ_URL = "https://github.com/karthiksiddu24/pcai-fraud-showcase/releases/download/lab-v1/transactions.tgz"

import subprocess
TGZ_PATH = f"{LAB_DIR}/raw/transactions.tgz"

if not os.path.exists(TGZ_PATH):
    print(f"Downloading {TRANSACTIONS_TGZ_URL} → {TGZ_PATH}")
    subprocess.run(["curl", "-fL", "-o", TGZ_PATH, TRANSACTIONS_TGZ_URL], check=True)
    print("Download complete.")
else:
    print(f"Already downloaded: {TGZ_PATH}")

# Show the file
subprocess.run(["ls", "-lh", TGZ_PATH])

In [ ]:
# Untar the dataset. After this we have card_transaction.v1.csv (~2.2 GB).
CSV_PATH = f"{LAB_DIR}/raw/card_transaction.v1.csv"

if not os.path.exists(CSV_PATH):
    print("Untarring (~30 sec)...")
    subprocess.run(["tar", "-xzf", TGZ_PATH, "-C", f"{LAB_DIR}/raw"], check=True)
    print("Done.")
else:
    print(f"Already extracted: {CSV_PATH}")

subprocess.run(["ls", "-lh", f"{LAB_DIR}/raw"])

import os
size_gb = os.path.getsize(CSV_PATH) / (1024**3)
with open(CSV_PATH) as f:
    header = f.readline().strip()
    first  = f.readline().strip()
print(f"\nCSV size:  {size_gb:.2f} GB")
print(f"Header:    {header}")
print(f"First row: {first}")

## Section 5 — Meet the data with pandas

Before training, let's look at the data the way any data scientist would: load a sample, look at the columns, get a feel for the fraud rate.

We use **pandas** here (CPU). Note that loading all 24M rows takes ~30 seconds; we'll load only a sample for the peek.

In [ ]:
import pandas as pd

# Read just the first 100k rows for the peek
df_sample = pd.read_csv(CSV_PATH, nrows=100_000)
print("Shape (of sample):", df_sample.shape)
df_sample.head()

In [ ]:
# What does the fraud label look like?
print("Fraud label distribution (in first 100k rows):")
print(df_sample['Is Fraud?'].value_counts())

print("\nA few fraud rows:")
df_sample[df_sample['Is Fraud?'] == 'Yes'].head(3)

## Section 6 — The RAPIDS speed-up moment 🚀

Now we'll demonstrate one of PCAI's headline value-props: **GPU-accelerated data processing with NVIDIA RAPIDS**.

The same operation — *"for every user, count how many transactions and what fraction were fraud"* — performed on:

- **pandas** (CPU, single-threaded): runs on your notebook pod's CPU cores
- **cuDF** (GPU, RAPIDS): runs on the L40s GPU attached to your pod

**This is on the full 24-million-row dataset, not a sample.** Watch the timing.

**Why this matters for the pitch:** any data scientist on PCAI can drop `import cudf as pd` into their notebook and get **10-30× speedups** on standard pandas operations. No code rewrite, no cluster setup. *That* is what the platform's NVIDIA-AI-Enterprise integration gives you.

> 💡 **Note:** when you import cuDF in the next cell, you may see a `libcudart.so: cannot open shared object file` warning from cuDF's optional Numba integration. **Ignore it — it's harmless.** cuDF uses its own C++ engine (libcudf) for everything you'll run; the missing library is only needed for an optional JIT compiler. The speedup numbers at the bottom of the cell prove cuDF is working correctly.

In [ ]:
import time
import pandas as pd

print("Reading 24M rows with pandas (CPU)...")
t0 = time.time()
df_pd = pd.read_csv(CSV_PATH, usecols=['User', 'Amount', 'Is Fraud?'])
t_read_pd = time.time() - t0
print(f"Read time: {t_read_pd:.1f} sec  ({len(df_pd):,} rows)")

print("\nGroup-by user → fraud rate, with pandas (CPU)...")
df_pd['is_fraud_int'] = (df_pd['Is Fraud?'] == 'Yes').astype(int)
t0 = time.time()
result_pd = df_pd.groupby('User').agg(
    tx_count=('Amount', 'count'),
    fraud_rate=('is_fraud_int', 'mean'),
).reset_index()
t_grp_pd = time.time() - t0
print(f"Group-by time: {t_grp_pd:.2f} sec")
print("\nResult (head):")
print(result_pd.head())

# Free the pandas DataFrame so cuDF run has fresh memory
del df_pd, result_pd

In [ ]:
# Suppress the benign libcudart / Numba-init UserWarning so the output is clean.
# cuDF works fine through libcudf; the warning is only about an optional JIT compiler.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import cudf, time

print("Reading 24M rows with cuDF (GPU)...")
t0 = time.time()
df_gpu = cudf.read_csv(CSV_PATH, usecols=['User', 'Amount', 'Is Fraud?'])
t_read_gpu = time.time() - t0
print(f"Read time: {t_read_gpu:.1f} sec  ({len(df_gpu):,} rows)")

print("\nGroup-by user → fraud rate, with cuDF (GPU)...")
df_gpu['is_fraud_int'] = (df_gpu['Is Fraud?'] == 'Yes').astype('int32')
t0 = time.time()
result_gpu = df_gpu.groupby('User').agg(
    tx_count=('Amount', 'count'),
    fraud_rate=('is_fraud_int', 'mean'),
).reset_index()
t_grp_gpu = time.time() - t0
print(f"Group-by time: {t_grp_gpu:.2f} sec")
print("\nResult (head):")
print(result_gpu.head())

print("\n" + "=" * 50)
print(f"  pandas CSV read:     {t_read_pd:6.1f} sec")
print(f"  cuDF   CSV read:     {t_read_gpu:6.1f} sec   →  {t_read_pd/t_read_gpu:.1f}× speedup")
print(f"  pandas group-by:     {t_grp_pd:6.2f} sec")
print(f"  cuDF   group-by:     {t_grp_gpu:6.2f} sec   →  {t_grp_pd/t_grp_gpu:.1f}× speedup")
print("=" * 50)

# Free GPU memory before training Job runs
del df_gpu, result_gpu
import gc
_ = gc.collect()   # underscore = suppress the trailing "9" in cell output

## Section 7 — Get NVIDIA's preprocessing script

**We are now done with EDA. Let's start the actual ML pipeline.**

The next step is **preprocessing**: turning the raw CSV into a *graph-shaped* dataset. The fraud-detection model is a Graph Neural Network — it treats users as nodes, merchants as nodes, and every transaction as an *edge* between a user and a merchant. The preprocessing step builds that graph from the CSV.

NVIDIA's blueprint ships the preprocessing code at [github.com/NVIDIA-AI-Blueprints/financial-fraud-detection](https://github.com/NVIDIA-AI-Blueprints/financial-fraud-detection). We'll clone it into our working directory — the training Job will use this script directly.

**Layman tip:** we are running NVIDIA's published reference code, **unchanged**. The PCAI value-prop is *how easy the platform makes it to deploy that code* — not that we wrote any of it ourselves.

In [ ]:
import subprocess, os

REPO_DIR = f"{LAB_DIR}/scripts/financial-fraud-detection"

if not os.path.exists(REPO_DIR):
    print("Cloning NVIDIA's fraud-detection blueprint...")
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/NVIDIA-AI-Blueprints/financial-fraud-detection.git",
        REPO_DIR
    ], check=True)
    print("Clone complete.")
else:
    print(f"Already cloned: {REPO_DIR}")

subprocess.run(["ls", f"{REPO_DIR}/src/"])

## Section 8 — Give Kubernetes your NGC API key

Now we get into the production MLOps pattern. We're going to train the model by **submitting a Kubernetes Job** that:

1. Pulls NVIDIA's official training container image (`nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0`)
2. Runs preprocessing followed by training inside that container, with our data mounted in
3. Exits when done, leaving the trained model files in our shared storage

To pull NVIDIA's image from `nvcr.io`, the cluster needs credentials. NVIDIA provides these as an **NGC API key** — free, generated at [ngc.nvidia.com](https://ngc.nvidia.com).

**Your instructor will give you an NGC API key** for this lab. Paste it into the cell below, then run the cell — it creates a Kubernetes Secret in your namespace called `ngc-secret`. The training Job we submit next will reference this secret.

**Layman tip:** A Kubernetes **Secret** is just a credentialed object stored in your namespace. Your `default-editor` service account is allowed to create them inside your own namespace — no admin involvement needed. This is one of the small but huge PCAI conveniences: you can do real MLOps work without an admin in the loop.

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  PASTE YOUR NGC API KEY HERE             ║
# ╚══════════════════════════════════════════╝

NGC_API_KEY = "nvapi-PASTE-YOUR-NGC-KEY-HERE"

# ──────────────────────────────────────────────
assert NGC_API_KEY.startswith("nvapi-"), "NGC API key should start with 'nvapi-'"
assert len(NGC_API_KEY) > 30, "Looks too short — check the key you pasted."
print(f"NGC key set (length {len(NGC_API_KEY)} chars). Creating ngc-secret in namespace {NS}...")

In [ ]:
import subprocess

# Delete any pre-existing ngc-secret in this namespace (safe; ignored if absent)
subprocess.run([
    "kubectl", "delete", "secret", "ngc-secret", "-n", NS, "--ignore-not-found"
], check=False)

# Create the secret
result = subprocess.run([
    "kubectl", "create", "secret", "docker-registry", "ngc-secret",
    "--docker-server=nvcr.io",
    "--docker-username=$oauthtoken",
    f"--docker-password={NGC_API_KEY}",
    "--docker-email=lab@example.com",
    "-n", NS,
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Failed to create ngc-secret.")

# Verify
subprocess.run(["kubectl", "get", "secret", "ngc-secret", "-n", NS,
                "-o", "jsonpath={.metadata.name} ({.type})\n"])

# Wipe the key from memory now that the secret exists in the cluster
NGC_API_KEY = "<consumed>"
print("\nngc-secret created. NGC_API_KEY variable cleared from memory.")

## Section 9 — Configure the training

The training image expects a JSON config at `/app/config.json`. It specifies:

- **Where to read input data from** (`/data` inside the container — we'll mount the preprocessed graph data there)
- **Where to write output to** (`/trained_models` — we'll mount your `trained_models/` directory there)
- **The model architecture**: `GNN_XGBoost` — a 2-hop GraphSAGE neural network plus an XGBoost classifier head
- **Hyperparameters**: hidden channels, learning rate, batch size, number of epochs, etc.

These are NVIDIA's recommended defaults from the blueprint. We use them unchanged.

**Stretch task (optional, after you finish the lab):** come back to this cell, change `n_hops` to 3 or `learning_rate` to 0.1, re-run the Job, and log it as a second MLflow run. You'll see two side-by-side runs in MLflow with different F1 scores.

In [ ]:
import json, os

training_config = {
    "paths": {
        "data_dir":   "/data",
        "output_dir": "/trained_models",
    },
    "models": [
        {
            "kind": "GNN_XGBoost",
            "gpu":  "single",
            "hyperparameters": {
                "gnn": {
                    "hidden_channels": 32,
                    "n_hops":          2,
                    "layer":           "SAGEConv",
                    "dropout_prob":    0.1,
                    "batch_size":      4096,
                    "fan_out":         10,
                    "num_epochs":      8,
                },
                "xgb": {
                    "max_depth":         6,
                    "learning_rate":     0.2,
                    "num_parallel_tree": 3,
                    "num_boost_round":   512,
                    "gamma":             0.0,
                },
            },
        }
    ],
}

CONFIG_PATH = f"{LAB_DIR}/training_config.json"
with open(CONFIG_PATH, 'w') as f:
    json.dump(training_config, f, indent=2)

print(f"Wrote {CONFIG_PATH}")
print(json.dumps(training_config, indent=2))

## Section 10 — Build the training Job manifest

Here is the production MLOps pattern. We're going to declare a **Kubernetes Job** that:

- Uses NVIDIA's published training image
- Requests 1 GPU
- Mounts the shared PVC at `/mnt/shared` (so it can read your raw data, write outputs)
- Runs a bash script that does **preprocessing then training**, back-to-back
- Cleans itself up after completion (`ttlSecondsAfterFinished: 600`)

**Layman tip:** A Kubernetes **Job** is a one-shot workload. It creates a pod, the pod runs to completion, exits. Compare to a Deployment which runs forever. This is exactly the right primitive for a batch training run.

**Why preprocessing + training in one Job?** Two reasons:
1. Both steps need the same image (NVIDIA's container has both the cuGraph/cuDF dependencies AND `/app/main.py`)
2. Simpler for a lab — one Job to watch, one stream of logs

In production you'd split them (preprocessing daily, training weekly), but for the lab they go together.

**This cell just builds the YAML and prints it for you to see. The next cell submits it.**

In [ ]:
JOB_NAME = f"fraud-train-student-{STUDENT_ID}"

JOB_YAML = f"""
apiVersion: batch/v1
kind: Job
metadata:
  name: {JOB_NAME}
  namespace: {NS}
  labels:
    app: fraud-lab
    student: "{STUDENT_ID}"
spec:
  ttlSecondsAfterFinished: 600
  backoffLimit: 0
  template:
    metadata:
      labels:
        app: fraud-lab
        student: "{STUDENT_ID}"
      annotations:
        sidecar.istio.io/inject: "false"
    spec:
      restartPolicy: Never
      imagePullSecrets:
        - name: ngc-secret
      volumes:
        - name: shared-volume
          persistentVolumeClaim:
            claimName: kubeflow-shared-pvc
      containers:
        - name: trainer
          image: nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0
          imagePullPolicy: IfNotPresent
          env:
            - name: PYTHONUNBUFFERED
              value: "1"
            - name: STUDENT_ID
              value: "{STUDENT_ID}"
          resources:
            limits:
              nvidia.com/gpu: 1
              memory: 32Gi
              cpu: 8
            requests:
              memory: 16Gi
              cpu: 4
          volumeMounts:
            - name: shared-volume
              mountPath: /mnt/shared
          command: ["bash", "-lc"]
          args:
            - |
              set -ex
              echo "=========================================="
              echo "[lab] Training container started"
              echo "[lab] Pod node:  $(hostname)"
              echo "[lab] Student:   ${{STUDENT_ID}}"
              echo "=========================================="
              nvidia-smi --query-gpu=name,memory.total --format=csv
              
              LAB_DIR=/mnt/shared/lab-student-${{STUDENT_ID}}
              
              echo ""
              echo "[lab] === STEP 1/3: install category_encoders (missing from base image) ==="
              pip install --no-cache-dir -q category_encoders 2>&1 | tail -3
              python -c "from category_encoders import BinaryEncoder; print('[lab] category_encoders OK')"
              
              echo ""
              echo "[lab] === STEP 2/3: preprocessing — CSV → graph dataset (~5 min) ==="
              python << 'PYEOF'
              import sys, os, json
              STUDENT_ID = os.environ['STUDENT_ID']
              LAB_DIR = f'/mnt/shared/lab-student-{{STUDENT_ID}}'
              sys.path.insert(0, f'{{LAB_DIR}}/scripts/financial-fraud-detection/src')
              from preprocess_TabFormer_lp import preprocess_data
              user_mask, mx_mask, tx_mask = preprocess_data(LAB_DIR)
              with open(f'{{LAB_DIR}}/feature_masks.json', 'w') as f:
                  json.dump({{'user': user_mask, 'merchant': mx_mask, 'user_to_merchant': tx_mask}},
                            f, indent=2, default=int)
              print('[lab] Preprocessing complete. Outputs at', LAB_DIR)
              PYEOF
              
              echo ""
              echo "[lab] === STEP 3/3: training — GNN + XGBoost (~40 sec on L40s) ==="
              # Set up the paths the training script expects
              mkdir -p $LAB_DIR/trained_models
              cp $LAB_DIR/training_config.json /app/config.json
              # Point /data at our preprocessed graph and /trained_models at our output dir
              # by symlinking — main.py reads /data and /trained_models from config
              rm -rf /data /trained_models 2>/dev/null || true
              ln -sf $LAB_DIR/gnn /data
              ln -sf $LAB_DIR/trained_models /trained_models
              
              cd /
              python /app/main.py
              
              echo ""
              echo "=========================================="
              echo "[lab] DONE — outputs:"
              ls -la $LAB_DIR/trained_models/python_backend_model_repository/prediction_and_shapley/1/ || true
              echo "=========================================="
"""

print(JOB_YAML)

# Write to disk so we can apply it
JOB_YAML_PATH = f"{LAB_DIR}/training-job.yaml"
with open(JOB_YAML_PATH, 'w') as f:
    f.write(JOB_YAML)
print(f"\nJob manifest saved to: {JOB_YAML_PATH}")

## Section 11 — Submit the Job and watch it run

Now the big moment. We submit the Job to Kubernetes and watch what happens.

The cell below does three things:

1. **Submits the Job** with `kubectl apply`
2. **Polls the pod state every 10 sec** while the pod is Pending or ContainerCreating — prints the latest event so you can see *image pull progress*, *container creation*, etc.
3. **Once the pod is Running**, switches to **streaming the container's stdout** into this cell

**What you'll see, in order:**

- `Pending → ContainerCreating` (just a moment)
- `Pulling image nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0` ← this is where the 5-10 min wait happens *for the first student on each GPU node*. After that, the image is cached on the node and subsequent students skip this step.
- `Successfully pulled image`
- `Started container`
- Then the actual container logs: `nvidia-smi` output, the pip install, the preprocessing print statements, the GraphSAGE training output (`Epoch 0, validation f1: 0.5722` …), early stopping, `Training Completed.`

**Sit back and watch.** This is the moment the trainer will talk through with the room. Image pull → image cache → container start → preprocessing → training → outputs on shared storage. End to end, no YAML you wrote yourself.

In [ ]:
import subprocess, time

# Delete any previous run with the same Job name (idempotency)
subprocess.run(["kubectl", "delete", "job", JOB_NAME, "-n", NS, "--ignore-not-found"],
               check=False, capture_output=True)
time.sleep(2)

# Submit
print(f"[{time.strftime('%H:%M:%S')}] Submitting Job: {JOB_NAME}")
r = subprocess.run(["kubectl", "apply", "-f", JOB_YAML_PATH, "-n", NS],
                   capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("Job submission failed.")

In [ ]:
import subprocess, time

print(f"[{time.strftime('%H:%M:%S')}] Waiting for pod to start...\n")
print("(First student per GPU node: ~5-10 min image pull. After that, ~10 sec.)\n")

last_event = ""
for _ in range(180):  # max 30 min of polling
    # Get pod phase
    p = subprocess.run(
        ["kubectl", "get", "pods", "-n", NS,
         "-l", f"job-name={JOB_NAME}",
         "-o", "jsonpath={.items[0].status.phase}"],
        capture_output=True, text=True
    )
    phase = p.stdout.strip() or "(no pod yet)"

    # Get the most recent pod-related event message (image pull progress, etc.)
    e = subprocess.run(
        ["kubectl", "get", "events", "-n", NS,
         "--field-selector", "involvedObject.kind=Pod",
         "--sort-by=.lastTimestamp",
         "-o", "jsonpath={.items[-1].message}"],
        capture_output=True, text=True
    )
    event = e.stdout.strip()

    if event and event != last_event:
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] {phase}: {event}")
        last_event = event

    if phase == "Running":
        print(f"\n[{time.strftime('%H:%M:%S')}] Pod Running — streaming container logs below ↓\n")
        break
    if phase in ("Failed", "Succeeded"):
        print(f"\n[{time.strftime('%H:%M:%S')}] Pod terminated before reaching Running: {phase}")
        break

    time.sleep(10)

# Stream logs until container exits
proc = subprocess.Popen(
    ["kubectl", "logs", "-f", f"job/{JOB_NAME}", "-n", NS],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, text=True
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

# Final Job status
print("\n--- Job status ---")
subprocess.run(["kubectl", "get", "job", JOB_NAME, "-n", NS])

## Section 12 — Verify the trained model

If you see `Training Completed.` in the logs above, the Job succeeded and your trained model is sitting on the shared PVC. Let's confirm.

**What you should see:** six files under `python_backend_model_repository/prediction_and_shapley/1/`:

| File | What it is |
|---|---|
| `model.py` | Triton Python backend code (~24 KB) — this is what runs at inference time |
| `state_dict_gnn_model.pth` | PyTorch weights for the GraphSAGE GNN (~64 KB) |
| `embedding_based_xgboost.json` | The XGBoost classifier (~3 MB) — trained on the GNN's embeddings |
| `meta.json` | Metadata about the model (~500 bytes) |
| `json_loader_writer.py` | Helper for I/O |
| `../config.pbtxt` | Triton's model config (~2 KB) |

Together, those six files are everything HPE MLIS needs to serve your model. **This is exactly the directory that gets uploaded to the MLIS Packaged Model.**

In [ ]:
import subprocess, os

MODEL_REPO = f"{LAB_DIR}/trained_models/python_backend_model_repository"

if not os.path.isdir(MODEL_REPO):
    print("❌ Trained model directory not found. Check the Job logs above for errors.")
else:
    print(f"✅ Trained model found at: {MODEL_REPO}\n")
    subprocess.run(["find", MODEL_REPO, "-type", "f", "-printf", "%p\\t%s bytes\\n"])

## Section 13 — Set up MLflow auth

Now the **governance moment**. We log this training run — params, metrics, the entire trained-model directory — to MLflow. Anyone in the org with MLflow access can then:

- See *who* trained the model
- See *what* hyperparameters were used
- Download the *artifacts*
- Promote the model from `None` → `Staging` → `Production`
- Audit *every* stage change

**That's MLOps governance, free with PCAI.** Compare to writing your own tracking, your own S3 buckets, your own IAM dance on bare cloud.

**The 'auth' part is automatic.** PCAI mounts a JWT token at `/etc/secrets/ezua/.auth_token` into every notebook pod. We use that token as:
- The MLflow REST API auth (`MLFLOW_TRACKING_TOKEN`)
- The local-s3 access key (`AWS_ACCESS_KEY_ID`) — with the special string `"s3"` as the secret

One token, both services. Zero config from you.


In [ ]:
import os

# Read PCAI's auth token (auto-mounted into every notebook pod)
with open('/etc/secrets/ezua/.auth_token') as f:
    token = f.read().strip()

# Wire up MLflow + boto3 (S3) with the token
os.environ['MLFLOW_TRACKING_URI']    = MLFLOW_URI
os.environ['MLFLOW_TRACKING_TOKEN']  = token
os.environ['MLFLOW_S3_ENDPOINT_URL'] = S3_ENDPOINT
os.environ['AWS_ACCESS_KEY_ID']      = token
os.environ['AWS_SECRET_ACCESS_KEY']  = 's3'

print("Auth configured:")
print(f"  MLFLOW_TRACKING_URI    = {MLFLOW_URI}")
print(f"  MLFLOW_TRACKING_TOKEN  = (length {len(token)} chars)")
print(f"  MLFLOW_S3_ENDPOINT_URL = {S3_ENDPOINT}")
print("  AWS_ACCESS_KEY_ID      = (mirrors the token)")
print("  AWS_SECRET_ACCESS_KEY  = 's3'  (PCAI's local-s3 convention)")

# Quick sanity check — confirm we can reach MLflow
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
print(f"\nConnected to MLflow. Search experiments: {len(mlflow.search_experiments())} found in your view.")

## Section 14 — Log everything to MLflow as `student-N-fraud-detection`

Now we open an MLflow run named after you and log:

- **Parameters**: every hyperparameter from the training config (`gnn_hidden_channels`, `xgb_learning_rate`, etc.)
- **Tags**: the dataset name, the training image we used, our pod hostname, today's date
- **Artifacts**: the entire `python_backend_model_repository/` directory uploaded to local-s3
- **Registered model**: `student-N-fraud-detection` version 1, pointing at this run

This cell takes ~30 sec to ~1 min (mostly the artifact upload to local-s3).

In [ ]:
import mlflow, datetime, socket

EXPERIMENT_NAME = f"student-{STUDENT_ID}-fraud-detection"
MODEL_NAME      = f"student-{STUDENT_ID}-fraud-detection"
RUN_NAME        = f"hands-on-lab-run-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"Run ID: {run.info.run_id}")
    print(f"Experiment: {EXPERIMENT_NAME}")

    # Tags — searchable metadata about the run
    mlflow.set_tags({
        "student_id":      str(STUDENT_ID),
        "dataset":         "IBM TabFormer (synthetic credit-card transactions)",
        "training_image":  "nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0",
        "compute":         "1x NVIDIA L40s (PCAI Job pod)",
        "notebook_pod":    socket.gethostname(),
        "lab_session":     "hands-on-lab",
    })

    # Parameters — hyperparameters flattened from training_config.json
    hyperparams = training_config['models'][0]['hyperparameters']
    for k, v in hyperparams['gnn'].items():
        mlflow.log_param(f"gnn_{k}", v)
    for k, v in hyperparams['xgb'].items():
        mlflow.log_param(f"xgb_{k}", v)
    mlflow.log_param("model_kind", training_config['models'][0]['kind'])
    mlflow.log_param("gpu_mode",   training_config['models'][0]['gpu'])

    # Artifacts — upload the entire trained model directory to local-s3
    print(f"\nUploading model artifacts from {MODEL_REPO} ...")
    mlflow.log_artifacts(MODEL_REPO, artifact_path="python_backend_model_repository")
    print("Upload complete.")

    # Also save the training config and the feature mask file as artifacts
    mlflow.log_artifact(CONFIG_PATH, artifact_path=".")
    if os.path.exists(f"{LAB_DIR}/feature_masks.json"):
        mlflow.log_artifact(f"{LAB_DIR}/feature_masks.json", artifact_path=".")

    # Register the model so it appears in MLflow's Model Registry
    model_uri = f"runs:/{run.info.run_id}/python_backend_model_repository"
    registered = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
    print(f"\n✅ Registered model: {MODEL_NAME}  version {registered.version}")
    print(f"✅ Run URL: {MLFLOW_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")

    RUN_ID = run.info.run_id
    EXPERIMENT_ID = run.info.experiment_id

## Section 15 — Find your run in the MLflow UI

🎉 **You just trained, registered, and tracked a fraud-detection model on HPE PCAI.**

Now go look at it in the MLflow web UI:

1. In the PCAI left-nav, open **Tools & Frameworks**
2. Find the **MLflow** tile, click **Open**
3. In the MLflow UI, top-nav **Experiments** → find `student-N-fraud-detection` (where N is your number)
4. Click into your run. You'll see:
   - Your **parameters** in the left panel (`gnn_hidden_channels: 32`, etc.)
   - **Tags** including your dataset, training image, pod hostname
   - The **Artifacts** tree on the right — open `python_backend_model_repository/prediction_and_shapley/1/` and you'll see your `model.py`, `state_dict_gnn_model.pth`, etc.
5. Top-nav **Models** → find `student-N-fraud-detection` → version 1 is registered. Stage is `None`. You can promote to `Staging` from the UI.

**This is the same MLflow setup that the showcase team used.** When your instructor demos MLIS deployment at the end of the lab, they may pick *your* registered model and put it behind an HTTPS endpoint with two UI clicks. **That moment, in front of the room, is what 'governance-grade ML' looks like on PCAI.**

In [ ]:
print("=" * 60)
print("  YOUR LAB RESULT — share this with your instructor")
print("=" * 60)
print(f"  Student:            {STUDENT_ID}")
print(f"  Namespace:          {NS}")
print(f"  MLflow experiment:  {EXPERIMENT_NAME}")
print(f"  MLflow run ID:      {RUN_ID}")
print(f"  Registered model:   {MODEL_NAME}  v1")
print(f"  S3 artifact URI:    s3://mlflow.pcai1/{EXPERIMENT_ID}/{RUN_ID}/artifacts/python_backend_model_repository")
print(f"  MLflow URL:         {MLFLOW_URI}")
print("=" * 60)

## Section 16 — Tidy up (optional)

Your Job's pod has already been auto-cleaned by Kubernetes (the `ttlSecondsAfterFinished: 600` in the YAML). 

If you want to delete the Job record itself early, run the cell below. Otherwise just close the notebook — your instructor will tear down everything when the lab ends.

**Note:** your trained model in MLflow and on local-s3 **stays** — that's the whole point of the lab.


In [ ]:
# Optional cleanup — only run if you want to free the Job record immediately
import subprocess
subprocess.run(["kubectl", "delete", "job", JOB_NAME, "-n", NS, "--ignore-not-found"])
subprocess.run(["kubectl", "get", "jobs", "-n", NS, "-l", "app=fraud-lab"])

---

## Recap — what you just did, in plain English

You started this lab with nothing but an empty Kubeflow notebook pod and an NGC API key. Forty minutes later you have:

1. **A real model trained on real data** — NVIDIA's open-source GraphSAGE + XGBoost fraud-detection blueprint, trained on 24M synthetic credit-card transactions from IBM TabFormer.
2. **A registered, versioned model** in MLflow's Model Registry, named after you, that any colleague can find, audit, or promote to Production.
3. **A direct comparison of pandas vs cuDF**, showing the RAPIDS speedup PCAI gives any data scientist out of the box.
4. **A working knowledge of the production MLOps pattern** — write a Kubernetes Job, submit it, watch the pod pull NVIDIA's image, train, exit. *This is exactly how the showcase team trained the model that's deployed in MLIS.*

## The PCAI value-prop in 5 bullets (your take-home)

- **You used 1 platform** — Notebooks, GPU scheduling, shared storage, MLflow, local-s3. Six different services on AWS, all one platform here.
- **You wrote zero YAML by hand.** The Job manifest was built by the notebook; you never opened a `.yaml` file.
- **You used NVIDIA's official training image**, unchanged, pulled with one secret. *Any* NGC catalog blueprint deploys this same way.
- **You got governance for free.** Every parameter, every artifact, every promotion is auditable. No external IAM, no extra storage to set up.
- **The lab works the same on 1 GPU or 100 GPUs.** Submit more Jobs, the platform schedules. Submit a different model, same pattern.

## What happens next

**Your instructor will now demo HPE MLIS** — the managed inference service. They'll take *one of the models in this room* (maybe yours!), point MLIS at the registered model URI from MLflow, fill in a UI form, and within a few minutes there's a live HTTPS endpoint serving real-time fraud predictions with SHAP-based explanations.

**That's the connection.** What you just did is the *training* half of the MLOps story. MLIS is the *serving* half. Together, that's the end-to-end value chain PCAI is built around.

---

Thanks for participating. Questions are welcome. 👋